# 📈 Session 1 — Ask Questions to SEC Filings (RAG)

In the next 30 minutes you'll build a working question-answering system over **real
annual reports (10-K filings)** from three semiconductor companies: **NVIDIA, AMD, and Micron**.

Everything is scaffolded — you just run the cells and watch each stage of the pipeline
come alive. At the end, you'll ask it your own questions.

```
 EDGAR (raw 10-K HTML)                                 your question
        │                                                    │
        ▼                                                    ▼
   sec-parser  ──►  section-aware chunks  ──►  embeddings ──►  vector search (Chroma)
 (semantic tree)   (tagged company+section)     (BGE model)         │
                                                                    ▼
                                          Gemini answer  ◄──  top-k source chunks
                                          (with citations)          │
                                                                    ▼
                                                          FinBERT sentiment tags
```

> 💡 **Tip:** in Colab, `Runtime → Run all` now, then follow along — the model downloads
> take a couple of minutes and will finish while we talk.


In [1]:
%pip install -q sec-parser sec-downloader sentence-transformers chromadb transformers google-genai

Note: you may need to restart the kernel to use updated packages.


In [1]:

import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")  # sec-parser is chatty about 10-Ks (see note below)

TICKERS = ["NVDA", "AMD", "MU"]

# Fallback if EDGAR is unreachable (conference wifi!): cached copies in this repo.
GITHUB_REPO = "YOUR_GITHUB_USERNAME/finance-ai-rag-kg"  # ← update after pushing your fork

## Step 1 — Get the filings from SEC EDGAR

Every US public company files an annual report (**10-K**) with the SEC, and all of it is
free on [EDGAR](https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany). The `sec-downloader`
package finds the latest filing for a ticker and gives us its primary HTML document.

(EDGAR asks that automated tools identify themselves with a name + email — that's the
`Downloader(...)` arguments.)


In [2]:
def get_filing_html(ticker: str) -> str:
    # 1) cached copy, if you cloned the repo and ran scripts/download_filings.py
    for base in (Path("../data/filings"), Path("data/filings")):
        local = base / f"{ticker}_10-K.html"
        if local.exists():
            print(f"{ticker}: using cached copy from the repo")
            return local.read_text(encoding="utf-8")
    # 2) live from EDGAR
    try:
        from sec_downloader import Downloader

        dl = Downloader("AIFinanceSession", "attendee@example.com")
        html = dl.get_filing_html(ticker=ticker, form="10-K")
        if isinstance(html, bytes):
            html = html.decode("utf-8", errors="replace")
        print(f"{ticker}: downloaded latest 10-K from EDGAR ({len(html):,} chars)")
        return html
    # 3) cached copy from GitHub
    except Exception as err:
        import urllib.request

        print(f"{ticker}: EDGAR failed ({err}) — fetching cached copy from GitHub")
        url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/data/filings/{ticker}_10-K.html"
        return urllib.request.urlopen(url).read().decode("utf-8", errors="replace")


filings = {ticker: get_filing_html(ticker) for ticker in TICKERS}

NVDA: using cached copy from the repo
AMD: using cached copy from the repo
MU: using cached copy from the repo


## The pain point 🤕

Let's peek at what we actually downloaded. This is the *raw* HTML around NVIDIA's
"Item 1A — Risk Factors" section — the part an analyst would most want to read:


In [3]:
html = filings["NVDA"]
position = html.find("Item 1A")
print(html[position - 200 : position + 1300])

coration:underline"><a style="color:#0000ff;font-family:'NVIDIA Sans',sans-serif;font-size:9pt;font-weight:400;line-height:100%;text-decoration:underline" href="#i82ea215a7c1f4862b6518f1348ddc832_16">Item 1A.</a></span></div></td><td colspan="3" style="padding:2px 1pt;text-align:left;vertical-align:bottom"><div style="text-align:justify"><span style="color:#0000ff;font-family:'NVIDIA Sans',sans-serif;font-size:9pt;font-weight:400;line-height:100%;text-decoration:underline"><a style="color:#0000ff;font-family:'NVIDIA Sans',sans-serif;font-size:9pt;font-weight:400;line-height:100%;text-decoration:underline" href="#i82ea215a7c1f4862b6518f1348ddc832_16">Risk Factors</a></span></div></td><td colspan="3" style="padding:2px 1pt;text-align:left;vertical-align:bottom"><div style="text-align:right"><span style="color:#000000;font-family:'NVIDIA Sans',sans-serif;font-size:9pt;font-weight:400;line-height:100%">&#160;</span><span style="color:#0000ff;font-family:'NVIDIA Sans',sans-serif;font-size:9

😱 Nested tables, inline styles, page numbers, invisible spans... and every company
formats theirs differently. Naive text extraction loses the document structure —
and structure is exactly what makes a filing readable (Items, sections, subsections).

## Step 2 — `sec-parser`: from HTML soup to a semantic tree

[`sec-parser`](https://github.com/alphanome-ai/sec-parser) (by Alphanome) classifies each
piece of a filing into **semantic elements** — titles, text, tables, page headers — and
arranges them into a tree that mirrors the document's *visual* structure.

> ℹ️ Its officially supported form type is the 10-Q (quarterly report); 10-Ks use the same
> HTML conventions, and it handles our three annual reports well — that's why we silenced
> the warnings above.


In [4]:
import sec_parser as sp

elements, trees = {}, {}
for ticker in TICKERS:
    elements[ticker] = sp.Edgar10QParser().parse(filings[ticker])
    trees[ticker] = sp.TreeBuilder().build(elements[ticker])
    print(f"{ticker}: {len(elements[ticker])} semantic elements")

print("\n--- the same NVIDIA 10-K, as a semantic tree (first 40 lines) ---\n")
print("\n".join(sp.render(trees["NVDA"]).splitlines()[:40]))

NVDA: 544 semantic elements
AMD: 614 semantic elements
MU: 850 semantic elements

--- the same NVIDIA 10-K, as a semantic tree (first 40 lines) ---

TopSectionTitle: Part I
├── TopSectionTitle: Item 1. Business
│   ├── TitleElement: Our Company
│   │   └── TextElement: NVIDIA pioneered accelerated co...ated in Delaware in April 1998.
│   ├── TitleElement: Our Businesses
│   │   └── TextElement: We report our business results ...nterprise workstation graphics.
│   ├── TitleElement: Our Markets
│   │   └── TextElement: We specialize in markets where ... Visualization, and Automotive.
│   ├── TitleElement: Data Center
│   │   └── TextElement: The NVIDIA Data Center platform...omation and robotics solutions.
│   ├── TitleElement: Gaming
│   │   └── TextElement: Gaming is the largest entertain...ure for gaming and GeForce NOW.
│   ├── TitleElement: Professional Visualization
│   │   └── TextElement: We serve the Professional Visua...d in our Data Center solutions.
│   ├── TitleElement: Auto

✨ That's the whole document — titles, sections, and the text under each — recovered
from the HTML soup. This structure is what we'll feed the RAG pipeline.

## Step 3 — Section-aware chunking

Embedding models can only look at a limited window of text, so we must split each filing
into **chunks**. The lazy way is fixed windows of ~500 tokens — but those cut sentences in
half, mix unrelated topics, and lose *where* the text came from.

Because `sec-parser` gave us the document's own structure, we can do better: **one chunk
never crosses a section boundary**, and every chunk carries `{company, section}` metadata —
so answers can cite *"NVIDIA 10-K — Item 1A"* instead of *"chunk #217"*.


In [5]:
def chunk_filing(filing_elements, ticker, max_chars=2000, min_chars=200):
    """Group text under its nearest title; split anything longer than max_chars."""
    chunks, buffer, section = [], [], "Preamble"

    def flush():
        text = " ".join(buffer).strip()
        buffer.clear()
        while len(text) > max_chars:  # split long sections at a sentence boundary
            cut = text.rfind(". ", 0, max_chars)
            cut = cut + 1 if cut > min_chars else max_chars
            chunks.append({"company": ticker, "section": section, "text": text[:cut].strip()})
            text = text[cut:].strip()
        if len(text) >= min_chars:
            chunks.append({"company": ticker, "section": section, "text": text})

    for element in filing_elements:
        if isinstance(element, (sp.TopSectionTitle, sp.TitleElement)):
            title = element.text.strip()
            if title and not title.isdigit():  # bare page numbers sometimes sneak in as titles
                flush()
                section = title
        elif isinstance(element, sp.TextElement):
            buffer.append(element.text)
    flush()
    return chunks


all_chunks = []
for ticker in TICKERS:
    chunks = chunk_filing(elements[ticker], ticker)
    all_chunks.extend(chunks)
    print(f"{ticker}: {len(chunks)} chunks")
print(f"total: {len(all_chunks)} chunks")

sample = next(c for c in all_chunks if "export control" in c["text"].lower())
print(f"\n--- sample chunk · {sample['company']} · section: {sample['section'][:70]} ---")
print(sample["text"][:400], "...")

NVDA: 238 chunks
AMD: 253 chunks
MU: 227 chunks
total: 718 chunks

--- sample chunk · NVDA · section: Government Regulations ---
Our worldwide business activities are subject to various laws, rules, and regulations of the United States as well as of foreign governments.Over the past three years, we have been subject to a series of shifting and expanding export control restrictions, impacting our ability to serve customers outside the United States.In August 2022, the U.S. government, or USG, announced export restrictions an ...


## Step 4 — Embed the chunks and build the vector index

An **embedding model** turns text into a vector such that similar meanings land close
together — so "restrictions on selling chips to China" can find a chunk that says
"export licensing requirements" *even though they share no keywords*.

We use `bge-small-en-v1.5`, a strong small open model (fast even on CPU). On a GPU
runtime you can swap in `BAAI/bge-large-en-v1.5`, or a finance-tuned model like
`FinLang/finance-embeddings-investopedia` — one-line change.

The vectors go into [Chroma](https://www.trychroma.com/), an in-memory vector store —
no server to set up.


In [6]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

print(f"embedding {len(all_chunks)} chunks on {device} ...")
chunk_vectors = embedder.encode(
    [c["text"] for c in all_chunks],
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

import chromadb

collection = chromadb.Client().get_or_create_collection("filings")
collection.upsert(
    ids=[str(i) for i in range(len(all_chunks))],
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"company": c["company"], "section": c["section"][:120]} for c in all_chunks],
    embeddings=chunk_vectors.tolist(),
)
print(f"indexed {collection.count()} chunks ✅")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedding 718 chunks on mps ...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Step 5 — Retrieve + generate: the "G" in RAG

Now the classic RAG loop: embed the **question**, pull the closest chunks from the index,
and hand them to an LLM (**Gemini**) with strict instructions to answer *only* from those
excerpts, with citations.

🔑 Get a free Gemini API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
(1 minute, no credit card). In Colab, you can also store it once under the key icon 🔑 in the
left sidebar as a secret named `GEMINI_API_KEY`.


In [ ]:
import os


def get_gemini_key():
    if os.environ.get("GEMINI_API_KEY"):
        return os.environ["GEMINI_API_KEY"]
    try:  # Colab secret (🔑 icon in the left sidebar)
        from google.colab import userdata

        return userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    try:
        import getpass

        return getpass.getpass("Paste your Gemini API key (Enter to skip): ").strip()
    except Exception:
        return ""


GEMINI_API_KEY = get_gemini_key()
if GEMINI_API_KEY:
    from google import genai

    gemini = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini ready ✅")
else:
    gemini = None
    print("⚠️ No Gemini key — cells below will show the retrieved excerpts, just without a generated answer.")

In [ ]:
ANSWER_PROMPT = """You are a careful financial analyst. Answer the question using ONLY the
excerpts below, taken from SEC annual filings. Cite the excerpts you use like [1], [2].
If the excerpts do not contain the answer, say so plainly.

{context}

Question: {question}
Answer:"""


def retrieve(question, k=6):
    question_vector = embedder.encode([question], normalize_embeddings=True)
    result = collection.query(query_embeddings=question_vector.tolist(), n_results=k)
    return [{"text": doc, **meta} for doc, meta in zip(result["documents"][0], result["metadatas"][0])]


def ask(question, k=6):
    sources = retrieve(question, k)
    context = "\n\n".join(
        f"[{i + 1}] ({s['company']} 10-K — {s['section']})\n{s['text']}" for i, s in enumerate(sources)
    )
    print(f"❓ {question}\n")
    if gemini:
        response = gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=ANSWER_PROMPT.format(context=context, question=question),
        )
        print(response.text)
    else:
        print("(no Gemini key — retrieved excerpts below)")
    print("\n📄 sources:")
    for i, s in enumerate(sources):
        print(f"  [{i + 1}] {s['company']} — {s['section'][:75]}")
    return sources


_ = ask("What does NVIDIA say about its dependence on TSMC?")

In [ ]:
_ = ask("How is Micron's business exposed to China?")

## Step 6 — FinBERT: sentiment on the evidence

[FinBERT](https://huggingface.co/ProsusAI/finbert) is a BERT model fine-tuned on financial
text — it reads *"our revenue declined due to export restrictions"* as **negative** where a
generic sentiment model might shrug. We tag each retrieved source chunk, so you can see at
a glance whether the evidence behind an answer is good news or bad news.


In [ ]:
from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert", device=device)
FLAG = {"positive": "🟢", "negative": "🔴", "neutral": "⚪"}


def ask_with_sentiment(question, k=6):
    sources = ask(question, k)
    print("\n🎭 FinBERT sentiment of each source chunk:")
    ratings = finbert([s["text"] for s in sources], truncation=True, max_length=512)
    for i, (s, r) in enumerate(zip(sources, ratings)):
        print(f"  [{i + 1}] {FLAG[r['label']]} {r['label']:<9}({r['score']:.2f})  {s['company']} — {s['section'][:60]}")


ask_with_sentiment("What risks does AMD face from relying on third-party foundries?")

## 🎤 Your turn

Edit the question below and run the cell. Some ideas:
- *How does Micron describe demand for AI-related memory?*
- *What does AMD say about competition with NVIDIA?*
- *How do these companies use artificial intelligence in their own products?*
- *What happened to NVIDIA's business in China?*


In [ ]:
ask_with_sentiment("How does Micron describe demand for AI-related memory?")  # ← change me!

## One last question... 🧗

Our system is good at questions whose answer sits **inside one passage of one filing**.
Let's ask something different — a question about how these companies *relate to each other*:


In [ ]:
_ = ask("Which of these companies share exposure to export control risk, and how are the companies connected to each other through suppliers?")

### Why did that answer feel shallow?

Look at what retrieval gave the model: **k isolated text chunks, each from a single
document**. The model can summarize each excerpt, but nothing in the pipeline *knows* that:

- "restrictions on advanced computing chips" (NVIDIA), "export controls on our MI accelerators"
  (AMD) and "CAC review of our memory products" (Micron) are **the same underlying risk**;
- NVIDIA, AMD *and* Micron all quietly point to **the same Taiwanese foundry** in their
  supply-chain risk sections;
- a question about *connections between companies* is a question about **relationships**,
  not passages.

RAG retrieves *text*. It has no concept of **entities** (companies, risks, regulations) or
**edges** (discloses, depends on, subject to).

**Session 2:** we extract exactly those entities and relationships from the filings and load
them into a graph database — then ask this same question again and *see* the answer. 🕸️
